# Liu 2016 baseline — step-by-step notebook

这个 notebook 展示完整流程：

1. 设置项目环境和依赖路径
2. 加载 BLE CS 场景与段落定义
3. 读取并滤波多通道变量
4. 运行 Liu 2016 基线算法
5. 打印整体误差与单段诊断
6. 绘制并保存窗口级 BPM 曲线


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

project_root = Path('/Users/shenmeichen/26 X program/ble_hci_sensing-main')
if (project_root / "src").is_dir():
    sys.path.insert(0, str(project_root / "src"))
else:
    raise FileNotFoundError('Project root not found: /Users/shenmeichen/26 X program/ble_hci_sensing-main')

from ble_analysis.bootstrap import init_notebook
from ble_analysis.chfusion import ChFusionConfig, _overall_rel_error, load_multichannel_for_scenario
from ble_analysis.liu_2016 import run_liu_2016_benchmark
from ble_analysis.scenarios import load_scenario, print_scenario_summary
from ble_analysis.segments import BreathMetricParams, FilterParams

env = init_notebook(project_root)
project_root = env["project_root"]
FIGURES_DIR = env["FIGURES_DIR"]
REPORTS_DIR = env["REPORTS_DIR"]
CACHE_DIR = project_root / "outputs" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Python executable:", sys.executable)
print("sys.path[0]:", sys.path[0])

print("Project root:", project_root)
print("FIGURES_DIR:", FIGURES_DIR)
print("REPORTS_DIR:", REPORTS_DIR)
print("CACHE_DIR:", CACHE_DIR)


In [ ]:
# 加载场景并打印段落定义
scenario_id = 'cs_091339'
scenario = load_scenario(scenario_id, project_root=project_root)
print(f"Scenario: {scenario_id}")
print_scenario_summary(scenario)


In [ ]:
# 准备滤波与度量参数
filter_params = FilterParams()
metric_params = BreathMetricParams()
cfg = ChFusionConfig(
    breath_freq_low=metric_params.breath_freq_low,
    breath_freq_high=metric_params.breath_freq_high,
    window_length_sec=metric_params.window_length_sec,
    step_length_sec=metric_params.step_length_sec,
)

multichannel_by_var, fs, skipped = load_multichannel_for_scenario(
    scenario,
    project_root=project_root,
    filter_params=filter_params,
    cache_dir=str(CACHE_DIR),
    verbose=True,
)

print(f"fs={fs:.2f} Hz")
print("available variables:", list(multichannel_by_var.keys()))
print("cache skip status:", skipped)


In [ ]:
# 运行 Liu 2016 baseline 算法
bench = run_liu_2016_benchmark(
    None,
    scenario.segment_config,
    filter_params=filter_params,
    metric_params=metric_params,
    config=cfg,
    verbose=True,
    cache_dir=str(CACHE_DIR),
    multichannel_by_var=multichannel_by_var,
)

stats = _overall_rel_error(bench["results"], "liu_2016")
print("Liu 2016 cross-window mean error:")
print(stats)
print("segments processed:", sorted(bench["results"].keys()))


In [ ]:
# 打印单个段落结果与诊断
seg_name = '3'
row = bench["results"][seg_name]
print("Segment row keys:", list(row["liu_2016"].keys()))
print("Segment summary:", row["liu_2016"])

bpm_est = row["liu_2016"]["bpm_per_window"]
bpm_gt = row["bpm_gt"]
windows = np.arange(len(bpm_est))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(windows, bpm_est, marker="o", label="Liu-style BPM")
ax.axhline(bpm_gt, color="red", linestyle="--", label=f"GT {bpm_gt:.1f}")
ax.set_xlabel("Window index")
ax.set_ylabel("BPM")
ax.set_title(f"{scenario_id} segment {seg_name} — window-level BPM")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

fig_path = FIGURES_DIR / f"liu_2016_{scenario_id}_segment_{seg_name}.png"
fig.savefig(fig_path, dpi=200, bbox_inches="tight")
print(f"Saved figure: {fig_path}")
